In [1]:
#loading the toolbox, environment and dataset
import sys
sys.path.append("../src")

import pandas as pd

from gow_rag.data_loader import load_corpus, load_claims
from gow_rag.preprocessing import tokenize
from gow_rag.sparse_retrieval import SparseRetriever, count_words
from gow_rag.graph_of_words import build_edges, graph_term_weights
from gow_rag.evaluation import evaluate, evaluate_per_query
from gow_rag.paths import TABLES_DIR

corpus = load_corpus()
queries, qrels = load_claims("validation")
print("Papers:", len(corpus), "| Claims:", len(queries))

Papers: 5183 | Claims: 188


In [2]:
# Toy example: count vs indegree 
toy = "protein binds receptor and protein activates receptor while protein degrades"
toy_tokens = toy.split()

print("Arrows in the graph (window = 4):")
for before, after in sorted(build_edges(toy_tokens)):
    print("  ", before, "->", after)

comparison = pd.DataFrame({
    "count (TF)": count_words(toy_tokens),
    "indegree (TW)": graph_term_weights(toy_tokens),
}).sort_values("count (TF)", ascending=False)
comparison

Arrows in the graph (window = 4):
   activates -> protein
   activates -> receptor
   activates -> while
   and -> activates
   and -> protein
   and -> receptor
   binds -> and
   binds -> protein
   binds -> receptor
   protein -> activates
   protein -> and
   protein -> binds
   protein -> degrades
   protein -> receptor
   protein -> while
   receptor -> activates
   receptor -> and
   receptor -> degrades
   receptor -> protein
   receptor -> while
   while -> degrades
   while -> protein


,count (TF),indegree (TW)
protein,3,5
receptor,2,4
binds,1,1
and,1,3
activates,1,3
while,1,3
degrades,1,3


In [3]:
#Building the graph weights for all papers
doc_ids = corpus["doc_id"].tolist()
doc_tokens = [tokenize(text) for text in corpus["full_text"]]
doc_lengths = [len(tokens) for tokens in doc_tokens]

doc_counts = [count_words(tokens) for tokens in doc_tokens]            # bag-of-words
doc_graph_weights = [graph_term_weights(tokens) for tokens in doc_tokens]  # graph-of-words

n_edges = [len(build_edges(tokens)) for tokens in doc_tokens]
print("Average words per paper:   ", round(sum(doc_lengths) / len(doc_lengths), 1))
print("Average vertices per paper:", round(sum(len(w) for w in doc_graph_weights) / len(doc_ids), 1))
print("Average edges per paper:   ", round(sum(n_edges) / len(n_edges), 1))

Average words per paper:    147.7
Average vertices per paper: 98.6
Average edges per paper:    406.2


In [4]:
#creating the search engines for different methods
engines = {
    "TF-IDF": SparseRetriever(doc_ids, doc_counts, doc_lengths, model="tfidf"),
    "BM25": SparseRetriever(doc_ids, doc_counts, doc_lengths, model="bm25"),
    "TW-IDF": SparseRetriever(doc_ids, doc_graph_weights, doc_lengths, model="twidf"),
}
print("Ready:", list(engines))

Ready: ['TF-IDF', 'BM25', 'TW-IDF']


In [5]:
#results
runs = {name: engine.search_all(queries, tokenize) for name, engine in engines.items()}

results = pd.DataFrame({name: evaluate(run, qrels) for name, run in runs.items()}).T
results.index.name = "Method"
results

,MAP,P@10,Recall@10,MRR@10,nDCG@10
Method,,,,,
TF-IDF,0.7601,0.0957,0.8902,0.7636,0.7914
BM25,0.7704,0.0979,0.9115,0.7749,0.8049
TW-IDF,0.5999,0.0872,0.8215,0.6022,0.6518


In [6]:
#where we find disagreement between TW-IDF and BM25
per_query = {name: evaluate_per_query(run, qrels) for name, run in runs.items()}
difference = per_query["TW-IDF"]["AP"] - per_query["BM25"]["AP"]

print("TW-IDF better on:", (difference > 0).sum(), "claims")
print("BM25 better on:  ", (difference < 0).sum(), "claims")
print("Exactly equal on:", (difference == 0).sum(), "claims")

TW-IDF better on: 20 claims
BM25 better on:   80 claims
Exactly equal on: 88 claims


In [7]:
#saving the score
results.to_csv(TABLES_DIR / "twidf_replication_validation.csv")

all_scores = pd.concat(per_query, names=["Method"]).reset_index()
all_scores.to_csv(TABLES_DIR / "twidf_replication_validation_per_query.csv", index=False)

print("Saved 2 files in:", TABLES_DIR)

Saved 2 files in: C:\Users\Lenovo\Desktop\Projects for IP Paris Application\Graph of words (Reviewing the paper with dataset)\github_upload\scientific-literature-GOW-RAG\results\tables
